In [2]:
import pandas as pd

data = pd.read_csv('/drugLibTrain_raw.tsv', sep='\t')

print(data.shape)
data.head()

(3107, 9)


,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,3947,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,1951,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


In [17]:
# 1. Combine the 3 text columns into one

data['text'] = (data['benefitsReview'].fillna("")  +  " " + data['sideEffectsReview'].fillna("") + " " + data['commentsReview'].fillna(""))

# 2. Keep only clear ratings: 1-4 (bad) and 7-10 (good) -> removing 5 and 6 because they are in middle and confuse the model

data = data[(data['rating'] <=4) | (data['rating'] >= 7)]

# 3. Make the label: 1 = positive, 0 = negative -> (data['rating'] >=7) gives true and false, and astype turns it into 0 and 1

data['label'] = (data['rating'] >=7).astype(int)

print(data.shape)
print(data['label'].value_counts())



(2791, 11)
label
1    2130
0     661
Name: count, dtype: int64


In [19]:
print(data['text'].iloc[0])



mangagement of congestive heart failur cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness monitor blood pressure , weight and asses for resolution of fluid


In [34]:
import re #a tool for finding patterns in text
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words = stop_words - {"not", "no", "nor", "didn", "doesn", "wasn",
                           "weren", "couldn", "wouldn", "shouldn", "isn",
                           "aren", "don", "won", "hasn", "haven"} #keep these words

def clean_text(text):
  text = text.lower() #lowercase
  text = re.sub(r"[^a-z\s]", " ", text) #keep only letters and spaces
  words = text.split() #split into list of words
  words = [w for w in words if w not in stop_words] #remove stopwords
  return " ".join(words) #join back into one text

data['clean_text'] = data['text'].apply(clean_text)

print(data['text'].iloc[1])
print('-----')
print(data['clean_text'].iloc[1])


Although this type of birth control has more cons than pros, it did help with my cramps. It's also effective with the prevention of pregnancy. (Along with use of condoms as well) Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results. I Hate This Birth Control, I Would Not Suggest This To Anyone.
-----
although type birth control cons pros help cramps also effective prevention pregnancy along use condoms well heavy cycle cramps hot flashes fatigue long lasting cycles months concidering changing different bc first time using kind bc unfortunately due constant hassel not happy results hate birth control would not suggest anyone


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


X= the text, y = label , 1 or 0 the answer

test_size=0.2: 20% goes to the test set.

random_state=42: makes the split the same every time you run it. 42 is just a habit, and any number works.

stratify=data['label']: keeps the same mix of positive and negative in both parts. Remember you have 3× more positives.

max_features=5000: use only the 5,000 most useful words.

ngram_range=(1, 2): look at single words and word pairs. So “not happy” is learned as one idea, not just “not” and “happy” separately.

fit_transform on train: learn the words and convert them. transform on test: only convert, no learning. The model must never peek at the exam

In [35]:
#split the data into words and numbers

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

#Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42, stratify=data['label'])

# Turn words into numbers
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)
print(X_test_vec.shape)

(2232, 5000)
(559, 5000)


In [36]:
#train the model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#1. Create and train the model
model= LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_vec, y_train)

#2. Make predicitons on the test data
y_pred = model.predict(X_test_vec)

#3. Evaluate the results
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion matrix:", confusion_matrix(y_test, y_pred))

Accuracy: 0.8300536672629696
              precision    recall  f1-score   support

           0       0.64      0.64      0.64       132
           1       0.89      0.89      0.89       427

    accuracy                           0.83       559
   macro avg       0.76      0.77      0.77       559
weighted avg       0.83      0.83      0.83       559

Confusion matrix: [[ 85  47]
 [ 48 379]]


In [37]:
#which words the model thinks are most positive and most negative
import numpy as np

words = vectorizer.get_feature_names_out()   # the 5,000 words
scores = model.coef_[0]                       # the score for each word

top_pos = np.argsort(scores)[-15:][::-1]     # 15 highest scores
top_neg = np.argsort(scores)[:15]            # 15 lowest scores

print("MOST POSITIVE WORDS:")
for i in top_pos:
    print(f"{words[i]:20s} {scores[i]:.2f}")

print("\nMOST NEGATIVE WORDS:")
for i in top_neg:
    print(f"{words[i]:20s} {scores[i]:.2f}")

MOST POSITIVE WORDS:
no side              1.97
take                 1.97
able                 1.59
years                1.58
morning              1.55
worked               1.41
sometimes            1.39
within               1.29
less                 1.27
needed               1.26
mild                 1.19
life                 1.17
used                 1.16
without              1.15
reduced              1.14

MOST NEGATIVE WORDS:
severe               -3.00
worse                -2.35
never                -2.11
horrible             -2.04
stopped              -1.93
told                 -1.90
drug                 -1.83
help                 -1.76
not                  -1.75
doctor               -1.74
extreme              -1.65
could not            -1.63
vomiting             -1.60
no benefits          -1.59
stopped taking       -1.57


In [38]:
my_reviews = ["this medicine worked great and no side effects",
              "terrible headaches would not recommend this to anyone"]

cleaned = [clean_text(r) for r in my_reviews]
print(model.predict(vectorizer.transform(cleaned)))

[1 0]


In [39]:
for w in ["help", "not help", "didn help", "helped"]:
    if w in vectorizer.vocabulary_:
        print(w, round(scores[vectorizer.vocabulary_[w]], 2))
    else:
        print(w, "-> not in the 5,000 words")

help -1.76
not help -0.12
didn help -0.06
helped 0.96


In [41]:
#check in the test file
test = pd.read_csv("/drugLibTest_raw.tsv", sep="\t")

#same steps as before : combine, filter, label, clean

test['text'] = (test['benefitsReview'].fillna("") + " " +
                test['sideEffectsReview'].fillna("") + " " +
                test['commentsReview'].fillna(""))

test = test[(test['rating'] <= 4) | (test['rating'] >=7 )]

test['label'] = (test['rating'] >= 7).astype(int)

test['clean_text'] = test['text'].apply(clean_text)

# Turn words into numbers (transform only, NO fit!)
X_final = vectorizer.transform(test['clean_text'])

#Predict and check
y_final = model.predict(X_final)
print(test.shape)
print("Accuracy:", accuracy_score(test['label'], y_final))
print(classification_report(test['label'], y_final))
print(confusion_matrix(test['label'], y_final))

(911, 12)
Accuracy: 0.8441273326015367
              precision    recall  f1-score   support

           0       0.71      0.69      0.70       241
           1       0.89      0.90      0.89       670

    accuracy                           0.84       911
   macro avg       0.80      0.79      0.80       911
weighted avg       0.84      0.84      0.84       911

[[166  75]
 [ 67 603]]
